# NB10.1 — Shortened Method Comparison with 29 Merge Points

This notebook compares four coefficient methods against the baseline $\lambda=p$
on the RS-PPO adapters. Phase A still searches the complete set of 75 preferences,
but the expensive ArmoRM Phase B uses a fixed preregistered pilot design with
exactly 29 unique merge points:

- 11 named baseline points;
- 6 Avg points;
- 6 MaxMin points with $c=0.5$;
- 6 Fair points with $\alpha=1,\varepsilon=0.05$;
- no additional Cert points, because Cert is certificate-only.

The six non-baseline preferences are the five `*_dominant` preferences and
`balanced`. Dirichlet points remain in the reward-free Phase-A diagnostics but are
not merge points.

| Label | Method | Formula / Problem |
|---|---|---|
| **Baseline** | no correction | $\lambda=p$ |
| **Avg** | preference alignment with trust region | $\arg\max_{\lambda\in\Delta} p^\top R\lambda-\rho(\lambda-p)^\top R(\lambda-p)$ |
| **Cert** | dual stationary cone certificate | $A\lambda=b,\ \lambda\ge0$; for $R>0$ this gives $\lambda=p$ |
| **MaxMin** | robust criterion with displacement | $\max_{\lambda\in\Delta}\min_k(R\lambda)_k$ subject to $\|\lambda-p\|_2\ge c\|e_{j^\star}-p\|_2$ |
| **Fair** | alpha-fair gain over baseline | $\max_{\lambda\in\Delta,\ R(\lambda-p)\ge-\varepsilon}\sum_i u_\alpha((R(\lambda-p))_i+\varepsilon)$ |

No method is defined in this notebook. All implementations are imported from
`src/`; duplicate implementations are a preregistration error.

## Two-Phase Design

| Phase | What happens | ArmoRM? | Repeatable? |
|---|---|---:|---:|
| **A** | compute coefficients for all methods and all 75 preferences, feasibility checks, movement diagnostics | no | yes |
| **B** | evaluate the fixed 29-point shortlist on 80 prompts | yes | once |

Phase A takes only seconds. Phase B is the preregistered one-shot phase. Any
ArmoRM contact without a valid preregistration consumes the evaluation; the gate
therefore fails closed.

## Primary Comparison Metric

The primary score is the **raw preference-weighted sum**

$$U_p(\lambda)=p^\top r(\lambda),$$

where $r(\lambda)$ is the five-dimensional ArmoRM reward vector. This is a
measurement definition, not an identity with the proxy $p^\top R\lambda$.
Rank and min-max normalization on the same reward matrix are reported only as
sensitivity analyses. They require no additional evaluation.

## Cert Row

Cert is reported as a **certificate**, not as an additional reward measurement.
NB09.1 found $t^\star=0$ for all 77 historical preferences under both matrices.
The dual certificate therefore gives $\lambda=p$ exactly and is included in the
final table without an additional merge evaluation.

## Two Method Properties That Must Be Visible

1. **MaxMin has an empty feasible set at simplex vertices for $c<1$.** This is
   recorded as `INFEASIBLE_EMPTY_SET`, not as collapse.
2. **Fair is preference-blind on the interior of the simplex.** Fair is therefore
   treated as an orthogonal fairness diagnostic, not as a general $f(p,R)$ answer
   to RQ1.

**Cell 2**

## 1. Repository

In [ ]:
# Cell 3
%cd /content
import os, shutil, zipfile
from pathlib import Path

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

# One-click setup for the RS-PPO adapters that are not stored in the repository.
# If the ZIP is already under /content, it is used directly; otherwise
# exactly one Colab upload dialog appears when this cell runs.
adapter_root = Path(repo_path) / "results/rs_ppo_armorm_circular/rs_runs"
adapter_axes = ("helpfulness", "correctness", "coherence", "complexity", "verbosity")
adapter_files = [
    adapter_root / f"ppo_{axis}/adapter/{name}"
    for axis in adapter_axes
    for name in ("adapter_config.json", "adapter_model.safetensors")
]

if not all(path.is_file() for path in adapter_files):
    candidates = sorted(Path("/content").glob("rs_ppo_armorm_adapters*.zip"))
    if not candidates:
        from google.colab import files
        print("Please select rs_ppo_armorm_adapters.zip now.")
        uploaded = files.upload()
        candidates = [Path.cwd() / name for name in uploaded if name.lower().endswith(".zip")]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one adapter ZIP, found: {candidates}")
    archive = candidates[0].resolve()
    adapter_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        target = adapter_root.resolve()
        for member in bundle.infolist():
            destination = (target / member.filename).resolve()
            if target not in destination.parents and destination != target:
                raise RuntimeError(f"Unsafe path in adapter ZIP: {member.filename}")
        bundle.extractall(target)

missing_adapters = [str(path.relative_to(Path(repo_path))) for path in adapter_files if not path.is_file()]
if missing_adapters:
    raise FileNotFoundError(f"Adapter ZIP is incomplete; missing: {missing_adapters}")
print(f"[OK] {len(adapter_axes)} RS-PPO-Adapter available.")

**Cell 4**

## 2. Runtime and Dependencies

In [ ]:
# Cell 5
!nvidia-smi || echo "No GPU — Phase A will still run."

In [ ]:
# Cell 6
# Python <= 3.12: the same stack as NB08. Python 3.13: the next
# compatible stack with prebuilt wheels; tokenizers 0.19.1, pandas 2.2.2, and
# numpy < 2.1 do not have compatible Linux wheels there.
import sys
print(f"Python {sys.version.split()[0]}")
if sys.version_info >= (3, 13):
    !pip install -q -U --only-binary=:all: "pandas==2.2.3" "numpy==2.1.3" "scipy==1.14.1" "protobuf==5.29.5" "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" "accelerate==1.1.1" datasets pyyaml bitsandbytes safetensors huggingface_hub
else:
    !pip install -q -U "pandas==2.2.2" "numpy<2.1" "protobuf>=5.29.1,<6.0.0" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" scipy datasets pyyaml bitsandbytes safetensors huggingface_hub
!python -c "import platform, transformers, tokenizers, peft, accelerate, bitsandbytes; print('Runtime:', platform.python_version(), 'transformers', transformers.__version__, 'tokenizers', tokenizers.__version__, 'peft', peft.__version__, 'accelerate', accelerate.__version__, 'bitsandbytes', bitsandbytes.__version__)"

**Cell 7**

## 3. Settings

In [ ]:
# Cell 8
from __future__ import annotations

import gc, json, hashlib, sys, zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/master-thesis").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_TAG = "nb10_1_pilot29_run1"

# --- Inputs (identical to NB06.1 so the provenance chain remains intact) ------------
CONFIG_PATH       = PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml"
COSINE_MATRIX_CSV = PROJECT_ROOT / "results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_cos.csv"
GRAM_MATRIX_CSV   = PROJECT_ROOT / "results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_gram.csv"

# --- Outputs ---------------------------------------------------------------
RESULTS_DIR  = PROJECT_ROOT / "results" / f"nb10_method_comparison_{RUN_TAG}"
LAMBDA_CSV   = RESULTS_DIR / "lambda_table.csv"
PREREG_JSON  = RESULTS_DIR / "nb10_preregistration.json"
REWARD_CACHE = RESULTS_DIR / "reward_cache.jsonl"
FINAL_CSV    = RESULTS_DIR / "method_comparison.csv"
ROBUST_CSV   = RESULTS_DIR / "normalization_robustness.csv"
REPORT_JSON  = RESULTS_DIR / "nb10_report.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# NB09.1 run1: matrices_agree = True, primary_matrix = R_cos.
PRIMARY_MATRIX = "R_cos"

# --- Method parameters (Phase A is repeatable; freeze BEFORE the gate) --
RHO_AVG    = 0.5                                # NB06.1: RHO_M1_PENALTY
CERT_C     = 0.5                                # Cert trust region
CERT_EPS   = 1e-8
C_GRID     = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]   # MaxMin: 0 aggressive, >=1 certificate
ALPHA_GRID = [0.0, 1.0, 2.0]                    # Fair: 0 utilitarian, 1 Nash
EPS_GRID   = [0.02, 0.05, 0.10]                 # Fair: floor relaxation

# --- Phase B ----------------------------------------------------------------
RUN_REWARD_COLLECTION = False   # remains False until the gate is open
REWARD_PROMPT_PATH    = RESULTS_DIR / "nb10_reward_prompts.jsonl"
MAX_NEW_TOKENS        = 256
REPETITION_PENALTY    = 1.15
NO_REPEAT_NGRAM_SIZE  = 5
LAMBDA_DEDUP_DECIMALS = 8       # identical to src.lambda_utils.lambda_key

print(f"RUN_TAG         = {RUN_TAG}")
print(f"Results directory  = {RESULTS_DIR}")
print(f"Primary matrix = {PRIMARY_MATRIX}")
print(f"Reward-Phase    = {'ACTIVE' if RUN_REWARD_COLLECTION else 'off (Phase A only)'}")

**Cell 9**

## 4. Imports from `src/`

The methods imported here are **not** redefined in the notebook:

| Notebook name | Implementation | Thesis notation |
|---|---|---|
| Avg | `coefficient_portfolio.avg` | $\lambda^{\mathrm{Avg}}_\rho$ |
| Cert | `coefficient_portfolio.cert` | $\lambda^{\mathrm{Cert}}$ |
| MaxMin | `coefficient_methods.maxmin_coefficients` | $\lambda^{\mathrm{MaxMin}}_c$ |
| Fair | `coefficient_methods.fair_coefficients` | $\lambda^{\mathrm{Fair}}_{\alpha,\varepsilon}$ |

The mappings from legacy names to canonical names are located only in
`src/coefficient_portfolio.py`. Merge arithmetic, reward-cache utilities and metrics
come from `src/proxy_validation.py`; the canonical coefficient keys from
`src/lambda_utils.py`.

Gate G6 additionally searches the entire `.ipynb` file for duplicate function
definitions. An inline copy of a method is therefore a real gate violation, not
just a stylistic problem.

In [ ]:
# Cell 10
from src.experiment_config import get_attribute_order, load_experiment_config, validate_preference_vectors
from src.proxy_validation import (
    build_search_set,
    coefficient_key,
    collect_reward_matrix,
    load_labeled_matrix_csv,
    normalization_agreement,
    preference_utility,
    write_json,
)
from src.coefficient_portfolio import (
    avg,                 # = m1_plus, v13 nomenclature
    cert,                # = c1_plus_plus
    fair_alpha_eps,
    floor_lp_at_p,
    improvements,
    maxmin_c,
    maxmin_center,
)
from src.lambda_utils import lambda_key

print("Portfolio imported. No method is defined in this notebook.")

**Cell 11**

## 5. Configuration, $R$ Matrices, and Preferences

In [ ]:
# Cell 12
config      = load_experiment_config(CONFIG_PATH)
ATTRIBUTES  = get_attribute_order(config)
PREFERENCES = validate_preference_vectors(config)
m           = len(ATTRIBUTES)

# Matrices are resolved and loaded by regime in Section 5a+. Do
# NOT load them in advance: the path constants from Section 2 do not identify a regime.
R_cos = R_gram = R = None
eigenvalues = None


**Cell 13**

## 6. Provenance — Regime, Adapters, Matrices

The most expensive silent error in this experiment would be to merge adapters from
one regime and correct them using geometry from another regime. The notebook would
run and produce meaningless values. This cell therefore fails before any GPU time
is spent if the required artifacts do not agree.

In [ ]:
# Cell 14
REGIME = "rs_ppo"          # "rs_ppo" or "sft" — determines both adapters AND matrices

# 8-bit exactly mirrors the PPO run's reward path (ppo_log: armorm_precision=8bit).
# This is required for the upper-bound argument: the evaluator must be the same as
# the training signal. bf16 would be more precise but would deliberately create a different
# evaluation path; in that case enter "bfloat16" here AND report the deviation in Chapter 6.
ARMORM_PRECISION = "8bit"

# (1) Preferences must come from a single source. The config and src/preferences.py
#     are two copies; if they differ, the notebook measures something different from the rest
#     of the project.
from src.preferences import PREFERENCES as PREFERENCES_MODULE

_config_names, _module_names = set(PREFERENCES), set(PREFERENCES_MODULE)
if _config_names != _module_names:
    raise AssertionError(
        "Preferences differ.\n"
        f"  only in the config:          {sorted(_config_names - _module_names)}\n"
        f"  only in src/preferences.py:  {sorted(_module_names - _config_names)}\n"
        "One of the two copies is stale. Reconcile them before anything runs."
    )
for _name in sorted(_config_names):
    if not np.allclose(np.asarray(PREFERENCES[_name], float),
                       np.asarray(PREFERENCES_MODULE[_name], float), atol=1e-12):
        raise AssertionError(f"Preference {_name!r} has different values in the config and module.")
print(f"[OK] Preferences consistent: {len(_config_names)} entries, {sorted(_config_names)}")

# (2) Resolve adapters. RS-PPO runs are stored under the NB08 paths and SFT adapters
#     under adapters/. Nothing is guessed: execution stops unless one layout is complete.
ADAPTER_LAYOUTS = {
    "rs_ppo": [
        "results/rs_ppo_armorm_circular/rs_runs/ppo_{axis}/adapter",
        "results/rs_ppo_armorm_circular/rs_runs/ppo_{axis}",
    ],
    "sft": [
        str(config.get("adapter_dir", "adapters")) + "/tinyllama-helpsteer2-{axis}-adapter",
        "adapters/tinyllama-helpsteer2-{axis}-adapter",
    ],
}

def _resolve_adapters(regime):
    for pattern in ADAPTER_LAYOUTS[regime]:
        paths = {a: (PROJECT_ROOT / pattern.format(axis=a)).resolve() for a in ATTRIBUTES}
        if all((p / "adapter_config.json").is_file() for p in paths.values()):
            return pattern, paths
    tried = "\n".join("  " + p for p in ADAPTER_LAYOUTS[regime])
    raise FileNotFoundError(
        f"No complete adapter set found for regime {regime!r}. Checked:\n{tried}\n"
        "Correct the path instead of allowing the notebook to fall back to another regime."
    )

ADAPTER_PATTERN, ADAPTER_PATHS = _resolve_adapters(REGIME)
print(f"[OK] Adapter ({REGIME}): {ADAPTER_PATTERN}")
for _a, _p in ADAPTER_PATHS.items():
    print(f"       {_a:12s} {_p.relative_to(PROJECT_ROOT)}")

# (3) Resolve matrices by regime. The filename is NOT evidence: the
#     existing results/tinyllama_helpsteer2_R/ path does not identify the run from which the
#     matrix originates. It is therefore selected explicitly by regime here, and its
#     hash is included in the binding evidence.
MATRIX_LAYOUTS = {
    "rs_ppo": [
        ("results/nb09_1_geometry_run1/R_cos.csv",
         "results/nb09_1_geometry_run1/R_gram.csv"),
        ("results/rs_ppo_armorm_circular/geometry/R_cos.csv",
         "results/rs_ppo_armorm_circular/geometry/R_gram.csv"),
        ("results/rs_ppo_armorm_circular/nb09_1_R_cos.csv",
         "results/rs_ppo_armorm_circular/nb09_1_R_gram.csv"),
    ],
    "sft": [
        ("results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_cos.csv",
         "results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_gram.csv"),
    ],
}

def _resolve_matrices(regime):
    for cos_rel, gram_rel in MATRIX_LAYOUTS[regime]:
        cos_path, gram_path = PROJECT_ROOT / cos_rel, PROJECT_ROOT / gram_rel
        if cos_path.is_file() and gram_path.is_file():
            return cos_path.resolve(), gram_path.resolve()
    tried = "\n".join(f"  {c}\n  {g}" for c, g in MATRIX_LAYOUTS[regime])
    raise FileNotFoundError(
        f"No R matrices found for regime {regime!r}. Checked:\n{tried}\n"
        "Copy the RS-PPO matrices computed by NB09.1 there instead of falling back to a "
        "path whose name does not identify its regime."
    )

COSINE_MATRIX_CSV, GRAM_MATRIX_CSV = _resolve_matrices(REGIME)
R_cos = load_labeled_matrix_csv(COSINE_MATRIX_CSV, ATTRIBUTES)
R_gram = load_labeled_matrix_csv(GRAM_MATRIX_CSV, ATTRIBUTES)
R = R_cos if PRIMARY_MATRIX == "R_cos" else R_gram
_eig = np.linalg.eigvalsh(R)
assert _eig.min() > 0, f"R is not positive definite (minimum eigenvalue {_eig.min():.3e})."
print(f"[OK] Matrices ({REGIME}): {COSINE_MATRIX_CSV.relative_to(PROJECT_ROOT)}")
print(f"       dominant share {_eig.max() / _eig.sum() * 100:.2f} % "
      f"(RS-PPO expected ~35.1 %, SFT ~46 %)")

eigenvalues = _eig
offdiag = R[np.triu_indices(m, 1)]
print(f"Attribute: {list(ATTRIBUTES)}")
print(f"Eigenvalues {PRIMARY_MATRIX}: {np.round(eigenvalues, 4)}")
print(f"Off-diagonal entries: {offdiag.min():.4f} .. {offdiag.max():.4f} "
      f"(mean {offdiag.mean():.4f})")

# Pi0(R 1) != 0 is the condition under which Fair escapes collapse (Prop 26, v13.2).
# It would be violated for perfectly equicorrelated R, and Fair would return p.
escape = R @ np.ones(m)
escape = escape - escape.mean()
print(f"||Pi_0(R 1)|| = {np.linalg.norm(escape):.6f}  "
      f"({'Fair can escape' if np.linalg.norm(escape) > 1e-9 else 'DEGENERATE: Fair collapses'})")

display(pd.DataFrame(R, index=list(ATTRIBUTES), columns=list(ATTRIBUTES)).round(4))


**Cell 15**

### 6.1 Preference Set

The configuration contains exactly the 11 named preferences: five vertices, five
dominant preferences, and `balanced`. The complete search set contains 75 points:

- 11 named preferences;
- 64 deterministic Dirichlet points with
  `seed=20250225`, `alpha=1`, `min_component=0.02`.

The earlier NB09.1 preregistration used 13 named and 77 total preferences.
`quality_focused` and `detailed_answer` were removed from the final portfolio;
the 13/77 count is therefore historical. This deviation is recorded in the
preregistration and must be reported.

Phase A always uses all 75 preferences. The Dirichlet points remain useful for
diagnosing collapse and preference-blindness, but Phase B is deliberately limited
to the 11 named preferences.

In [ ]:
# Cell 16
# 11 named preferences + 64 Dirichlet points = 75 in Phase A.
# Phase B is later filtered explicitly to the 11 named preferences.
USE_FULL_SEARCH_SET = True

PREF_SET = [(name, np.asarray(vec, dtype=np.float64)) for name, vec in PREFERENCES.items()]

if USE_FULL_SEARCH_SET:
    seen = {tuple(np.round(v, 9)) for _, v in PREF_SET}
    B = build_search_set(m, n_dirichlet=64, dirichlet_alpha=1.0,
                         preferences=list(PREFERENCES.values()), seed=137)
    for j, vec in enumerate(B):
        key = tuple(np.round(vec, 9))
        if key not in seen:
            seen.add(key)
            PREF_SET.append((f"dirichlet_{j:02d}", np.asarray(vec, dtype=np.float64)))

for name, p in PREF_SET:
    assert abs(p.sum() - 1.0) < 1e-9 and np.all(p >= -1e-12), f"{name} is not in the simplex."
assert len(PREF_SET) == 75, "NB10.1 expects 75 preferences in Phase A."

INTERIOR = [name for name, p in PREF_SET if np.all(p > 1e-12)]
print(f"|P| = {len(PREF_SET)}  (including {len(INTERIOR)} interior, {len(PREF_SET) - len(INTERIOR)} on the boundary)")
display(pd.DataFrame([p for _, p in PREF_SET],
                     index=[n for n, _ in PREF_SET], columns=list(ATTRIBUTES)).round(4))

**Cell 17**

### 6.2 Phase-B Subset

Phase A is reward-free and computes all points for the complete search set.
Phase B evaluates only a preregistered subset with ArmoRM.

The subset is defined by a **rule**, not by inspecting rewards:

1. all named preferences;
2. for each named preference and method, every valid Phase-A candidate;
3. identical coefficient vectors are deduplicated only during reward collection.

This ensures that no reward result can influence the choice of evaluated point.

In [ ]:
# Cell 18
PHASE_B_RULE = ("29-point pilot: 11 named baselines; Avg on all named preferences; "
                "MaxMin(c=0.5) and Fair(alpha=1,eps=0.05) on the six interior "
                "named preferences; Cert is the baseline certificate")
PREF_SET_B = [(name, p) for name, p in PREF_SET if name in PREFERENCES]
PHASE_B_NAMES = {name for name, _ in PREF_SET_B}
PILOT_INTERIOR_NAMES = {name for name, p in PREF_SET_B if np.all(p > 1e-12)}
PILOT_MAXMIN_C = 0.5
PILOT_FAIR_ALPHA = 1.0
PILOT_FAIR_EPS = 0.05

assert len(PREF_SET_B) == len(PREFERENCES), (
    f"PREF_SET_B has {len(PREF_SET_B)} entries, PREFERENCES {len(PREFERENCES)}."
)
assert PHASE_B_NAMES, "The Phase-B preference set is empty."
_expected_pilot_names = {"balanced"} | {f"dominant_{a}" for a in ATTRIBUTES}
assert PILOT_INTERIOR_NAMES == _expected_pilot_names, (
    f"Unexpected interior pilot preferences: {sorted(PILOT_INTERIOR_NAMES)}"
)
print(f"Phase A: |P| = {len(PREF_SET)}    named baselines: {len(PREF_SET_B)}")
print("Pilot method points:", sorted(PILOT_INTERIOR_NAMES))


**Cell 19**

## 7. Phase A — Compute All $\lambda$ (Reward-Free)

This phase takes seconds and may be repeated as often as needed before the gate is
closed.

In [ ]:
# Cell 20
def norm_R(v):
    return float(np.sqrt(max(v @ R @ v, 0.0)))


rows = []
for pname, p in PREF_SET:
    base = {"p_name": pname, **{f"p_{a}": float(p[i]) for i, a in enumerate(ATTRIBUTES)}}

    results = []

    lam = avg(p, R, RHO_AVG)
    results.append(("Avg", f"rho={RHO_AVG}", lam, "OK", {}))

    lam_cert, t_cert = cert(p, R, CERT_C, CERT_EPS)
    t_lp, collapsed = floor_lp_at_p(R, p)
    results.append(("Cert", "", lam_cert,
                    "FLOOR_COLLAPSED" if collapsed else "FLOOR_NONTRIVIAL",
                    {"t_star": t_cert, "t_star_lp": t_lp}))

    for c in C_GRID:
        lam_mm, t_mm, status = maxmin_c(p, R, c)
        results.append(("MaxMin", f"c={c}", lam_mm, status, {"t_star": t_mm}))

    for alpha in ALPHA_GRID:
        for eps in EPS_GRID:
            lam_f, u_f, status = fair_alpha_eps(p, R, alpha, eps)
            results.append(("Fair", f"alpha={alpha},eps={eps}", lam_f, status, {"u_alpha": u_f}))

    for method, params, lam, status, extra in results:
        row = dict(base, method=method, params=params, status=status, **extra)
        if lam is None:
            row.update({f"lam_{a}": np.nan for a in ATTRIBUTES})
            row.update({"dist_l2": np.nan, "dist_R": np.nan, "proxy_pRlam": np.nan,
                        "min_delta": np.nan, "moved": False, "usable": False})
        else:
            v = lam - p
            row.update({f"lam_{a}": float(lam[i]) for i, a in enumerate(ATTRIBUTES)})
            row.update({"dist_l2": float(np.linalg.norm(v)),
                        "dist_R": norm_R(v),
                        "proxy_pRlam": float(p @ R @ lam),
                        "min_delta": float(np.min(improvements(p, R, lam))),
                        "moved": bool(np.linalg.norm(v) > 1e-8),
                        "usable": True})
        rows.append(row)

lam_df = pd.DataFrame(rows)
lam_df.to_csv(LAMBDA_CSV, index=False)
lam_cols = [f"lam_{a}" for a in ATTRIBUTES]
p_cols = [f"p_{a}" for a in ATTRIBUTES]
print(f"{len(lam_df)} lambda rows -> {LAMBDA_CSV}")
display(lam_df.head(20))

**Cell 21**

### 7.1 Movement Diagnostics

Two different phenomena must remain separate:

- **collapse**: a feasible method returns the baseline $\lambda=p$;
- **empty feasible set**: the specified optimization problem has no admissible
  point.

They are counted separately below.

In [ ]:
# Cell 22
key_series = lam_df["method"] + lam_df["params"].map(lambda s: f"({s})" if s else "")
summary = (lam_df.assign(key=key_series).groupby("key")
           .agg(n=("moved", "size"),
                n_usable=("usable", "sum"),
                n_moved=("moved", "sum"),
                mean_dist_l2=("dist_l2", "mean"),
                max_dist_l2=("dist_l2", "max"),
                mean_dist_R=("dist_R", "mean"))
           .sort_values("mean_dist_l2", ascending=False))
display(summary.round(4))

print("\nStatus distribution:")
display(lam_df["status"].value_counts().rename_axis("status").reset_index(name="n"))

unusable = lam_df[~lam_df["usable"]]
if len(unusable):
    print(f"\n{len(unusable)} rows without lambda:")
    display(unusable[["p_name", "method", "params", "status"]])
    n_infeasible = int((unusable["status"] == "INFEASIBLE_BALL_MISSES_SIMPLEX").sum())
    n_failed = len(unusable) - n_infeasible
    print(f"  including mathematically empty: {n_infeasible}   solver failures: {n_failed}")
    if n_failed:
        print("  WARNING: Solver failures are NOT a result and must be resolved before Phase B.")
else:
    print("\nAll rows have a usable lambda.")

**Cell 23**

### 7.2 Consistency with NB09.1 and Theory

The following assertions deliberately fail when a mathematically expected property
breaks. They do not fail merely because a result changed numerically.

In [ ]:
# Cell 24
# (1) Cert returns p for every preference while the floor is collapsed.
cert_rows = lam_df[lam_df["method"] == "Cert"]
assert not cert_rows["moved"].any(), "Cert moves, contradicting NB09.1 run1."
print(f"[OK] Cert returns p for all {len(cert_rows)} preferences. Status: {sorted(cert_rows['status'].unique())}")

# (2) MaxMin(c>=1) collapses to p (Prop 30(a)).
mm1 = lam_df[(lam_df["method"] == "MaxMin") & (lam_df["params"] == "c=1.0") & lam_df["usable"]]
if len(mm1):
    max_shift = float(np.nanmax(mm1["dist_l2"]))
    assert max_shift < 1e-6, f"MaxMin(c=1.0) moves (maximum {max_shift:.2e}) — violating Proposition 30(a)."
    print(f"[OK] MaxMin(c=1.0) collapses to p (max {max_shift:.2e}, {len(mm1)} preferences).")

# (3) MaxMin is empty at vertices for c<1 — this is not a solver artifact.
for pname, p in PREF_SET:
    if np.all(p > 1e-12):
        continue
    sub = lam_df[(lam_df["p_name"] == pname) & (lam_df["method"] == "MaxMin")]
    for _, r in sub.iterrows():
        c_value = float(r["params"].split("=")[1])
        if c_value < 1.0 - 1e-9:
            assert r["status"] == "INFEASIBLE_BALL_MISSES_SIMPLEX", \
                f"{pname}/c={c_value}: expected an empty set, received {r['status']}"
print("[OK] MaxMin at boundary preferences for c<1 is consistently reported as empty.")

# (4) Every moving point worsens at least one proxy axis (trivial floor).
movers = lam_df[lam_df["moved"] & lam_df["min_delta"].notna()]
violations = movers[movers["min_delta"] > 1e-9]
assert len(violations) == 0, f"{len(violations)} moving points without deterioration — the floor would not be trivial."
print(f"[OK] All {len(movers)} moving lambda vectors worsen at least one proxy axis.")

# (5) Fair is preference-blind AS LONG AS the simplex bounds are inactive.
#     The objective U_alpha(R v + eps) depends only on v; p enters only through
#     -p_i <= v_i <= 1-p_i. As long as none of these bounds is active, the
#     solution is identical for all p. Mathematically, 'p_i > 0' is not sufficient:
#     a Dirichlet point may be close enough to the boundary that lambda_i = 0 is active. Therefore
#     the claim is tested only among solutions whose bounds are actually inactive.
#     At alpha = 0 the objective is linear; this case is still reported only descriptively.
fair_interior = lam_df[(lam_df["method"] == "Fair") & lam_df["usable"] &
                       lam_df["p_name"].isin(INTERIOR)]
SIMPLEX_BOUND_TOL = 1e-7
blind_rows = []
for params, group in fair_interior.groupby("params"):
    shifts = group[lam_cols].to_numpy(float) - group[p_cols].to_numpy(float)
    lambdas = group[lam_cols].to_numpy(float)
    simplex_free = np.all((lambdas > SIMPLEX_BOUND_TOL) &
                          (lambdas < 1.0 - SIMPLEX_BOUND_TOL), axis=1)
    free_shifts = shifts[simplex_free]
    spread_all = float(np.abs(shifts - shifts[0]).max()) if len(shifts) > 1 else 0.0
    spread_free = (float(np.abs(free_shifts - free_shifts[0]).max())
                   if len(free_shifts) > 1 else 0.0)
    alpha_value = float(params.split(",")[0].split("=")[1])
    active = float(np.abs(shifts).max())
    blind_rows.append({"params": params, "alpha": alpha_value,
                       "n_interior_p": len(group),
                       "n_simplex_free": int(simplex_free.sum()),
                       "n_simplex_bound": int((~simplex_free).sum()),
                       "spread_all_interior_p": spread_all,
                       "spread_simplex_free": spread_free,
                       "max_abs_shift": active,
                       "blind_when_simplex_free": spread_free < 1e-5})
    if alpha_value >= 1.0 - 1e-9 and len(free_shifts) > 1:
        assert spread_free < 1e-5, (
            f"Fair({params}): Displacement varies even with inactive "
            f"simplex bounds by {spread_free:.2e}."
        )
display(pd.DataFrame(blind_rows).round(6))
print("[OK] Fair is preference-blind for alpha >= 1 when the simplex bounds are inactive.")
print("     Interior Dirichlet points near the boundary with lambda_i = 0 are reported separately.")
print("     At alpha = 0 the objective is linear and may have boundary or non-unique")
print("     solutions; p has an effect precisely when a simplex bound is active.")
print("     Chapter 3 must report both the preference-blindness AND its limit.")

# (6) At alpha = 0 the relaxation is fully exhausted: min_delta = -eps exactly.
#     This is bang-bang behavior, not a compromise, and produces a very
#     large step for large eps. This method setting requires explicit documentation.
fair0 = lam_df[(lam_df["method"] == "Fair") & lam_df["params"].str.startswith("alpha=0.0") &
               lam_df["usable"]]
if len(fair0):
    for params, group in fair0.groupby("params"):
        eps_value = float(params.split("eps=")[1])
        worst = float(group["min_delta"].min())
        print(f"     Fair({params}): min_delta = {worst:.4f} versus -eps = {-eps_value:.4f}, "
              f"max ||lambda-p|| = {group['dist_l2'].max():.4f}")

**Cell 25**

### 7.3 Phase-B Budget

Each unique merge point is evaluated exactly once. Cert and MaxMin points that
collapse to the baseline do not require additional model evaluations.

**Cell 26**

## 8. Evaluation Prompts

The prompts are generated **inside this notebook**, so that `n_prompts=80` in the
preregistration does not depend on a separately executed script.

The prompts are drawn only from the validation split. The RS-PPO training in this
project used the train split. In addition, all prompts already present in existing
result files are excluded. The exclusion summary is part of the provenance.

In [ ]:
# Cell 27
from src.eval_prompts import build_eval_prompt_file, ensure_nb06_prompt_files

N_EVAL_PROMPTS = 80
EVAL_PROMPT_SEED = 137

# Fail closed: these files MUST be found or construction stops. Drawing prompts
# because an exclusion file could not be located is exactly the error that this
# guard is intended to prevent. If project history confirms that one is truly absent,
# set ALLOW_MISSING_EXCLUSIONS = True AND report the gap.
ALLOW_MISSING_EXCLUSIONS = False

# The two historical NB06 files are absent from a fresh clone. They are
# using the frozen NB06.1 rule (validation split, seed 137, 80+80)
# existing files are validated and never overwritten.
NB06_PROMPT_SUMMARY = ensure_nb06_prompt_files(
    PROJECT_ROOT,
    dataset_name=str(config["dataset_name"]),
    split="validation",
    seed=EVAL_PROMPT_SEED,
    n_per_set=80,
)
print(json.dumps(NB06_PROMPT_SUMMARY, indent=2, ensure_ascii=False))

REWARD_PROMPT_PATH.parent.mkdir(parents=True, exist_ok=True)
PROMPT_SUMMARY = build_eval_prompt_file(
    REWARD_PROMPT_PATH,
    n=N_EVAL_PROMPTS,
    seed=EVAL_PROMPT_SEED,
    project_root=PROJECT_ROOT,
    allow_missing_exclusions=ALLOW_MISSING_EXCLUSIONS,
)
print(json.dumps(PROMPT_SUMMARY, indent=2, ensure_ascii=False))
if not PROMPT_SUMMARY.get("disjointness_verified", False):
    print("\nNOTE: Disjointness from earlier evaluations has NOT been established. "
          "Report it this way, not otherwise.")


In [ ]:
# Cell 28
eval_points, origins = [], {}

def _register(vec, origin):
    key = lambda_key(vec, decimals=LAMBDA_DEDUP_DECIMALS)
    if key not in origins:
        origins[key] = []
        eval_points.append(np.asarray(vec, dtype=np.float64))
    origins[key].append(origin)

# Predetermined 29-point shortlist. Phase A remains complete with all grids and
# all 75 preferences; only the expensive ArmoRM path is shortened.
lam_df_named = lam_df[lam_df["p_name"].isin(PHASE_B_NAMES)].copy()

def _pilot_row_selected(r):
    if r["method"] in {"Avg", "Cert"}:
        return True
    if r["p_name"] not in PILOT_INTERIOR_NAMES:
        return False
    if r["method"] == "MaxMin":
        return np.isclose(float(r["params"].split("=")[1]), PILOT_MAXMIN_C)
    if r["method"] == "Fair":
        alpha_text, eps_text = r["params"].split(",")
        return (np.isclose(float(alpha_text.split("=")[1]), PILOT_FAIR_ALPHA) and
                np.isclose(float(eps_text.split("=")[1]), PILOT_FAIR_EPS))
    return False

_pilot_mask = lam_df_named.apply(_pilot_row_selected, axis=1)
lam_df_B = lam_df_named[_pilot_mask].copy()
assert set(lam_df_B["p_name"].unique()).issubset(PHASE_B_NAMES), \
    "Dirichlet points must not produce merge points for Phase B."

# All 11 baselines are registered exactly once. Methods contribute only moving
# points; Cert and collapsed Avg rows therefore require no evaluation.
for pname, p in PREF_SET_B:
    _register(p, f"baseline:{pname}")
for _, r in lam_df_B.iterrows():
    if bool(r["usable"]) and bool(r["moved"]):
        _register(r[lam_cols].to_numpy(float), f"{r['method']}({r['params']}):{r['p_name']}")

EVAL_POINTS = np.asarray(eval_points, dtype=np.float64)
n_unique = len(EVAL_POINTS)
_moving_counts = (lam_df_B[lam_df_B["usable"] & lam_df_B["moved"]]
                  .groupby("method").size().to_dict())
_expected_new = {"Avg": 6, "Cert": 0, "MaxMin": 6, "Fair": 6}
for _method, _expected in _expected_new.items():
    assert int(_moving_counts.get(_method, 0)) == _expected, (
        f"{_method}: expected {_expected} new points, "
        f"found {_moving_counts.get(_method, 0)}."
    )
assert n_unique == 29, f"Pilot expected 29 unique merge points, found {n_unique}."

# Read the prompt count from the prompt file; do not guess it.
if REWARD_PROMPT_PATH.is_file():
    n_prompts = sum(1 for line in REWARD_PROMPT_PATH.read_text(encoding="utf-8").splitlines() if line.strip())
else:
    n_prompts = 80
    print(f"NOTE: {REWARD_PROMPT_PATH.name} does not exist yet; using {n_prompts} Prompts.")

print(f"Eindeutige merge points:   {n_unique}")
print(f"Prompts je Punkt:          {n_prompts}")
print(f"Generierungen gesamt:      {n_unique * n_prompts}")
print(f"Aufteilung:                11 Baseline + 6 Avg + 0 Cert + 6 MaxMin + 6 Fair")
print(f"Phase-B method rows:    {len(lam_df_B)}")
hours = n_unique * n_prompts * 2.5 / 3600
print(f"\nRough runtime (A100, 256 new tokens, ~2.5 s per generation + scoring): ~{hours:.1f} h")
print("  " + ("FITS in one session" if hours < 3 else
              "TOO LONG for one session — reduce the preference set or grid"))

In [ ]:
# Cell 29
# Request Phase B for the subsequent gate check.
RUN_REWARD_COLLECTION = True
print("Phase B requested — ArmoRM starts only if the gate is open.")


**Cell 30**

## 9. GATE — Preregistration

This is the deliberate one-shot gate. It freezes the measurement protocol before
any reward is evaluated. The SHA-256 hash binds the exact Phase-A coefficient table:
if a method or grid is changed later, the hash no longer matches.

Before running this cell with `RUN_REWARD_COLLECTION=True`, the following must be
resolved:

1. acute-angle question #6;
2. final adapter and matrix provenance;
3. Lingxiao's approval of the Phase-B subset and the metrics.

**Cell 31**

### 9.1 Binding Evidence

The `lambda_sha256` above binds only the coefficient table, that is, the reward-free
part of the analysis. By itself, it does **not** bind prompts, adapters, matrices,
scorer configuration, or generation parameters. Without the complete evidence
below, one could swap the prompts after the fact while retaining the same
`lambda_sha256`.

In [ ]:
# Cell 32
def _sha256_file(path):
    """Return the SHA256 of one file."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _sha256_dir(path, patterns=("*.safetensors", "*.bin", "adapter_config.json")):
    """Return a stable SHA256 over the relevant files of one adapter directory."""
    digest = hashlib.sha256()
    for pattern in patterns:
        for file_path in sorted(Path(path).glob(pattern)):
            digest.update(file_path.name.encode())
            digest.update(_sha256_file(file_path).encode())
    return digest.hexdigest()

def _runtime_versions():
    """Record the numerical software path in the frozen binding."""
    import platform
    from importlib import metadata

    packages = ("torch", "transformers", "tokenizers", "peft", "accelerate",
                "bitsandbytes", "numpy", "pandas", "scipy")
    return {"python": platform.python_version(),
            **{name: metadata.version(name) for name in packages}}

def _model_revisions():
    """Best-effort exact revisions of the two external models."""
    revisions = {}
    for label, repo in (("base_model", str(config["base_model_name"])),
                        ("armorm", "RLHFlow/ArmoRM-Llama3-8B-v0.1")):
        try:
            from huggingface_hub import HfApi

            revisions[label] = HfApi().model_info(repo).sha
        except Exception as error:                      # offline or no hub access
            revisions[label] = f"unresolved: {type(error).__name__}"
    return revisions

BINDING = {
    "regime": REGIME,
    "adapter_pattern": ADAPTER_PATTERN,
    "adapters_sha256": {a: _sha256_dir(p) for a, p in ADAPTER_PATHS.items()},
    "matrix_cos_sha256": _sha256_file(COSINE_MATRIX_CSV),
    "matrix_gram_sha256": _sha256_file(GRAM_MATRIX_CSV),
    "prompts_sha256": _sha256_file(REWARD_PROMPT_PATH),
    "scorer_sha256": _sha256_file(PROJECT_ROOT / "src" / "armorm_scorer.py"),
    "portfolio_sha256": _sha256_file(PROJECT_ROOT / "src" / "coefficient_portfolio.py"),
    "base_model_name": str(config["base_model_name"]),
    "armorm_precision": ARMORM_PRECISION,
    "attribute_order": list(ATTRIBUTES),
    "generation": {"max_new_tokens": MAX_NEW_TOKENS,
                   "repetition_penalty": REPETITION_PENALTY,
                   "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
                   "decoding": "greedy, do_sample=False, num_beams=1"},
    "grids": {"rho_avg": RHO_AVG, "cert_c": CERT_C, "cert_eps": CERT_EPS,
              "c_grid": list(C_GRID), "alpha_grid": list(ALPHA_GRID), "eps_grid": list(EPS_GRID)},
    "phase_b_shortlist": {
        "expected_unique_points": 29,
        "baseline_preferences": sorted(PHASE_B_NAMES),
        "method_preferences": sorted(PILOT_INTERIOR_NAMES),
        "avg_rho": RHO_AVG,
        "maxmin_c": PILOT_MAXMIN_C,
        "fair_alpha": PILOT_FAIR_ALPHA,
        "fair_eps": PILOT_FAIR_EPS,
    },
    "src_sha256": {name: _sha256_file(PROJECT_ROOT / "src" / name) for name in (
        "merge.py", "proxy_validation.py", "metrics.py", "lambda_utils.py",
        "armorm_objectives.py", "eval_prompts.py")},
    "model_revisions": _model_revisions(),
    "runtime_versions": _runtime_versions(),
}

# A single hash covers the complete evidence. It is stored in the reward cache so that
# an interrupted session cannot resume under changed conditions.
BINDING_SHA256 = hashlib.sha256(
    json.dumps(BINDING, sort_keys=True).encode()).hexdigest()
print(json.dumps(BINDING, indent=2))
print()
print(f"BINDING_SHA256 = {BINDING_SHA256}")


In [ ]:
# Cell 33
PREREG_CONFIRM = True    # One-click mode: write only if no preregistration exists yet

lambda_hash = hashlib.sha256(
    lam_df[["p_name", "method", "params", "status"] + lam_cols]
    .round(9).to_csv(index=False).encode()
).hexdigest()

prereg = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "NB10.1 shortened method comparison (29 merge points)",
    "lambda_table_sha256": lambda_hash,
    "primary_matrix": PRIMARY_MATRIX,
    "methods": {
        "Avg": {"source": "src.coefficient_portfolio.m1_plus", "rho": RHO_AVG},
        "Cert": {"source": "src.coefficient_portfolio.c1_plus_plus", "c": CERT_C, "eps": CERT_EPS,
                 "note": "certificate row: lambda = p by NB09.1 run1 (t*=0 for 77/77, "
                         "R^-1 1 > 0); not measured separately"},
        "MaxMin": {"source": "src.coefficient_portfolio.maxmin_c", "phase_a_c_grid": C_GRID,
                   "phase_b_c": PILOT_MAXMIN_C,
                   "phase_b_preferences": sorted(PILOT_INTERIOR_NAMES)},
        "Fair": {"source": "src.coefficient_portfolio.fair_alpha_eps",
                 "phase_a_alpha_grid": ALPHA_GRID, "phase_a_eps_grid": EPS_GRID,
                 "phase_b_alpha": PILOT_FAIR_ALPHA, "phase_b_eps": PILOT_FAIR_EPS,
                 "phase_b_preferences": sorted(PILOT_INTERIOR_NAMES),
                 "note": "objective depends on the displacement only; p enters through the "
                         "simplex bounds. NOT preference-aware in the sense of RQ1."},
    },
    "baseline": "lambda = p",
    "n_preferences_phase_a": len(PREF_SET),
    "n_preferences_phase_b": len(PREF_SET_B),
    "phase_b_preference_rule": PHASE_B_RULE,
    "preference_names": [n for n, _ in PREF_SET],
    "phase_b_preference_names": [n for n, _ in PREF_SET_B],
    "preference_set_note": (
        "11 named preferences. Deviates from the 13 pre-registered in NB09.1: "
        "quality_focused and detailed_answer removed, uniform renamed to balanced. "
        "Phase B is additionally restricted to the frozen 29-point pilot shortlist. "
        "Report both deviations explicitly."
    ),
    "scope": "shortened 29-point pilot; not a replacement for the full NB10 grid",
    "binding": BINDING,
    "binding_sha256": BINDING_SHA256,
    "prompt_provenance": PROMPT_SUMMARY,
    "reward_model": {
        "name": "RLHFlow/ArmoRM-Llama3-8B-v0.1",
        "precision": ARMORM_PRECISION,
        "batch_size": 1,
        "head_mapping": "src.armorm_objectives.helpsteer_head_indices (external golden sample)",
        "scoring_format": "apply_chat_template(user + assistant)",
    },
    "generation_format": "apply_chat_template(user, add_generation_prompt=True) - identical to NB08",
    "raw_scores_retained": True,
    "error_layer": "paired bootstrap over prompts, 10000 draws, alpha=0.05",
    "multiplicity": "Holm-Bonferroni over all shortlisted moving (method, preference) comparisons",
    "n_unique_merge_points": int(n_unique),
    "metric_primary": "raw preference-weighted sum U_p = sum_i p_i r_i; identity normalization r~_i := r_i",
    "metric_implementation": "src.proxy_validation.preference_utility(..., normalization='identity')",
    "metric_robustness": ["minmax", "rank"],
    "metric_note": "supersedes rank-U_p from NB06.1 metric set v2",
    "prompts": {"path": str(REWARD_PROMPT_PATH.relative_to(PROJECT_ROOT)),
                "n": int(n_prompts),
                "max_new_tokens": MAX_NEW_TOKENS,
                "repetition_penalty": REPETITION_PENALTY,
                "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
                "decoding": "greedy, do_sample=False, num_beams=1"},
    "decision_rule": (
        "Primary: sign and magnitude of Delta U_p = U_p(lambda) - U_p(p) per method. "
        "Confirmatory improvement requires Delta U_p > 0 and a Holm-adjusted p-value < 0.05 "
        "across all shortlisted method/preference comparisons; harm is defined analogously for Delta U_p "
        "< 0. The unadjusted percentile bootstrap CI is descriptive only. No threshold is "
        "lowered after seeing the numbers. Cert is reported as a certificate row with Delta "
        "U_p = 0 by construction. Infeasible MaxMin cells are reported as infeasible and are "
        "not counted as collapses."
        " Conclusions apply to the frozen 29-point pilot only, not to omitted grid settings."
    ),
    "armorm_role": (
        "REGIME-DEPENDENT. In the RS-PPO regime ArmoRM is frozen during evaluation but "
        "was ALSO the PPO reward model, so this evaluation is circular and supports only "
        "upper-bound and diagnostic claims, not held-out proxy validity. The read-only "
        "firewall claim holds for the SFT regime only, and there only pending the AKUT #6 "
        "audit. Do not restate the firewall as a global property of the thesis."
    ),
}

_unresolved_revisions = {
    name: revision for name, revision in BINDING["model_revisions"].items()
    if str(revision).startswith("unresolved:")
}
if PREREG_CONFIRM and not PREREG_JSON.exists() and _unresolved_revisions:
    raise RuntimeError(
        "Preregistration not written: Hugging Face revisions are unresolved: "
        f"{_unresolved_revisions}. Check the network and start Run all again."
    )

if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    print(f"Preregistration already exists ({frozen['created_utc']}) — not overwritten.")
    if frozen["lambda_table_sha256"] != lambda_hash:
        print("\n*** WARNING: the lambda table differs from the frozen version. ***")
        print(f"    frozen: {frozen['lambda_table_sha256'][:16]}")
        print(f"    current:     {lambda_hash[:16]}")
        print("    Phase B stops. remainingore the grid or choose a new RUN_TAG.")
elif PREREG_CONFIRM:
    write_json(PREREG_JSON, prereg)
    print(f"Preregistration written -> {PREREG_JSON}")
    print(f"lambda-Hash: {lambda_hash}")
else:
    print("PREREG_CONFIRM is False — nothing was written. Preview:")
    print(json.dumps(prereg, indent=2)[:2000])

In [ ]:
# Cell 34
GATE_OPEN = False
if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    _diffs = []
    if frozen.get("lambda_table_sha256") != lambda_hash:
        _diffs.append("lambda_table_sha256")
    for _k, _v in BINDING.items():
        if frozen.get("binding", {}).get(_k) != _v:
            _diffs.append(f"binding.{_k}")
    GATE_OPEN = not _diffs
    if GATE_OPEN:
        print("GATE OPEN — the preregistration matches the lambda table AND binding evidence.")
    else:
        print("GATE CLOSED — mismatch in: " + ", ".join(_diffs))
        print("  Something changed after freezing: prompts, adapters, matrix, "
              "scorer, or grid. Resolve the cause; do not simply freeze again.")
else:
    print("GATE CLOSED — no preregistration.")

if RUN_REWARD_COLLECTION and not GATE_OPEN:
    raise RuntimeError(
        "RUN_REWARD_COLLECTION is True, but the gate is closed. Without a frozen, "
        "matching preregistration, ArmoRM will not be accessed."
    )

**Cell 35**

## 10. Phase B — Reward Evaluation

### 10.1 Merge Arithmetic and Cache

The context manager temporarily writes
$\theta(\lambda)=\theta_0+\sum_i\lambda_i\Delta_i$ into the base model and
then restores $\theta_0$. Applying these deltas is destructive; reloading the
entire model for every point would be prohibitively expensive.

In [ ]:
# Cell 36
if RUN_REWARD_COLLECTION:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from src.merge import combine_effective_deltas, effective_deltas, resolve_base_module

    BASE_MODEL_NAME = str(config["base_model_name"])
    # Resolved and checked against the regime in Section 5a+. Do NOT infer it again from the
    # config: config["adapter_dir"] points to the SFT adapters.
    adapter_paths = dict(ADAPTER_PATHS)
    assert BINDING["adapters_sha256"] == {a: _sha256_dir(p) for a, p in adapter_paths.items()}, \
        "Adapter weights changed after the binding evidence was created."

    generation_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, torch_dtype=torch.float32, device_map="auto")
    base_model.eval()
    print("Base model loaded in float32 (bf16 corrupts the merge endpoints).")

    DELTAS = effective_deltas(adapter_paths)   # fp32 on CPU, once
    print(f"Effective deltas prepared for {len(next(iter(DELTAS.values())))} modules.")
else:
    print("RUN_REWARD_COLLECTION is False — Phase B skipped.")

In [ ]:
# Cell 37
if RUN_REWARD_COLLECTION:
    @contextmanager
    def merged_model(model, deltas_by_adapter, lam):
        """Apply sum_i lambda_i delta_i, then restore the original weights."""
        merged = combine_effective_deltas(lam, deltas_by_adapter)
        originals = {}
        try:
            for module_name, delta_cpu in merged.items():
                module = resolve_base_module(model, module_name)
                originals[module_name] = module.weight.detach().clone()
                update = (module.weight.detach().to(dtype=torch.float32)
                          + delta_cpu.to(device=module.weight.device, dtype=torch.float32))
                with torch.no_grad():
                    module.weight.copy_(update.to(dtype=module.weight.dtype))
            yield model
        finally:
            for module_name, original in originals.items():
                with torch.no_grad():
                    resolve_base_module(model, module_name).weight.copy_(original)
            del originals, merged
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def generate_answer(prompt: str) -> str:
        device = next(base_model.parameters()).device
        # NB08 format. The RS-PPO adapters were used with apply_chat_template and
        # add_generation_prompt=True; a different format would mean evaluating
        # out of distribution. add_special_tokens=False because the template
        # already contains the prefix; this is checked below in this cell.
        text = generation_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        encoded = generation_tokenizer(text, return_tensors="pt", add_special_tokens=False)
        input_ids = encoded["input_ids"].to(device)
        with torch.inference_mode():
            generated = base_model.generate(
                input_ids=input_ids,
                attention_mask=encoded["attention_mask"].to(device),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=generation_tokenizer.eos_token_id)
        return generation_tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

    _probe = generation_tokenizer.apply_chat_template(
        [{"role": "user", "content": "probe"}], tokenize=False, add_generation_prompt=True)
    _bos = generation_tokenizer.bos_token_id
    _ids = generation_tokenizer(_probe, add_special_tokens=False)["input_ids"]
    assert _bos is None or _ids.count(_bos) <= 1, "Duplicate BOS token in the generation prompt."
    print("Generation format (identical to NB08):")
    print(repr(_probe))
    print("Merge and generation routines are ready.")

In [ ]:
# Cell 38
if RUN_REWARD_COLLECTION:
    import logging
    import time
    from datetime import timedelta
    from zoneinfo import ZoneInfo
    from src.armorm_objectives import ARMORM_HELPSTEER_OBJECTIVE_NAMES
    from src.armorm_scorer import make_score_prompt_answer
    from src.proxy_validation import collect_reward_tensor
    from src.tinyllama_training_utils import load_reward_prompts
    _cell38_started = time.perf_counter()

    # bitsandbytes emits the same known BF16-to-FP16 conversion message for many
    # layers. Filter only this message; all other warnings remain visible.
    class _BnbCastMessageFilter(logging.Filter):
        def filter(self, record):
            return ("MatMul8bitLt: inputs will be cast from torch.bfloat16 "
                    "to float16 during quantization") not in record.getMessage()

    _bnb_logger = logging.getLogger("bitsandbytes.autograd._functions")
    if not any(getattr(f, "_nb10_bnb_cast_filter", False) for f in _bnb_logger.filters):
        _bnb_filter = _BnbCastMessageFilter()
        _bnb_filter._nb10_bnb_cast_filter = True
        _bnb_logger.addFilter(_bnb_filter)

    reward_prompts = load_reward_prompts(REWARD_PROMPT_PATH)
    assert len(reward_prompts) == n_prompts, "Prompt count differs from the preregistration."
    assert all(a in ARMORM_HELPSTEER_OBJECTIVE_NAMES for a in ATTRIBUTES), \
        "ATTRIBUTES contains an axis without an anchored ArmoRM head mapping."

    # 8-bit: identical to the PPO run's reward path. As a result, the golden sample passes
    # with the model card's loose 0.35 tolerance because int8 shifts the regression heads
    # slightly.
    score_prompt_answer, scorer = make_score_prompt_answer(
        dtype="bfloat16", load_in_8bit=(ARMORM_PRECISION == "8bit"))
    assert scorer.describe()["precision"] == ("int8" if ARMORM_PRECISION == "8bit" else "bfloat16")
    scorer.assert_golden_sample()          # BEFORE the first real scoring operation
    print("Scorer:", scorer.describe())
    print(f"[time] ArmoRM setup including golden test: "
          f"{time.perf_counter() - _cell38_started:.1f} s")

    def _format_duration(seconds):
        seconds = max(0, int(round(seconds)))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    # Time only uncached merge points. Mapping via
    # coefficient_key remains correct even after an interrupted run.
    _cached_keys = set()
    if REWARD_CACHE.is_file():
        for _line in REWARD_CACHE.read_text(encoding="utf-8").splitlines():
            if not _line.strip():
                continue
            try:
                _record = json.loads(_line)
            except json.JSONDecodeError:
                continue
            if "key" in _record:
                _cached_keys.add(str(_record["key"]))
    _eval_index_by_key = {coefficient_key(lam): i + 1
                          for i, lam in enumerate(EVAL_POINTS)}
    _cached_eval_count = sum(key in _cached_keys for key in _eval_index_by_key)
    _missing_initial = n_unique - _cached_eval_count
    _progress = {"started": time.perf_counter(), "new_done": 0}
    _nominal_remaining = _missing_initial * n_prompts * 2.5
    print(f"[time] Cell 38: {n_unique} merge points x {n_prompts} Prompts; "
          f"{_cached_eval_count} from cache, {_missing_initial} remaining.")
    print(f"[time] Initial remaining-time estimate at 2.5 s per prompt pair: "
          f"{_format_duration(_nominal_remaining)}.")

    def reward_of_lambda(lam):
        """Per-prompt ArmoRM rewards for one merge point, shape (n_prompts, m)."""
        _point_started = time.perf_counter()
        _key = coefficient_key(lam)
        _position = _eval_index_by_key[_key]
        _new_number = _progress["new_done"] + 1
        print(f"[time] Starting merge point {_position}/{n_unique} "
              f"(remaining point {_new_number}/{_missing_initial}).")
        with merged_model(base_model, DELTAS, lam):
            answers = [generate_answer(record["prompt"]) for record in reward_prompts]
        scores = np.asarray([
            score_prompt_answer(record["prompt"], answer, ATTRIBUTES)
            for record, answer in zip(reward_prompts, answers)
        ], dtype=np.float64)
        assert scores.shape == (len(reward_prompts), m), scores.shape
        _progress["new_done"] += 1
        _point_elapsed = time.perf_counter() - _point_started
        _reward_elapsed = time.perf_counter() - _progress["started"]
        _cell_elapsed = time.perf_counter() - _cell38_started
        _average = _reward_elapsed / _progress["new_done"]
        _remaining = _missing_initial - _progress["new_done"]
        _eta_seconds = _average * _remaining
        _finish = (datetime.now(ZoneInfo("Europe/Berlin")) + timedelta(seconds=_eta_seconds)).strftime("%d.%m. %H:%M %Z")
        print(f"[time] Completed {_progress['new_done']}/{_missing_initial} remaining | "
              f"last point {_format_duration(_point_elapsed)} | "
              f"reward time {_format_duration(_reward_elapsed)} | "
              f"cell time {_format_duration(_cell_elapsed)} | "
              f"remaining {_format_duration(_eta_seconds)} | expected {_finish}")
        return scores          # Do NOT average: the paired bootstrap requires raw values

    REWARD_TENSOR = collect_reward_tensor(
        EVAL_POINTS, reward_of_lambda, REWARD_CACHE,
        num_prompts=n_prompts, binding_sha256=BINDING_SHA256)
    REWARD_MATRIX = REWARD_TENSOR.mean(axis=1)     # identical to the earlier matrix
    np.save(RESULTS_DIR / f"nb10_{RUN_TAG}_reward_tensor.npy", REWARD_TENSOR)
    print(f"Reward tensor: {REWARD_TENSOR.shape}   Reward matrix: {REWARD_MATRIX.shape}")


**Cell 39**

## 11. Final Table

For every valid Phase-B point, the following are reported:

- the method and preference;
- the coefficient vector;
- $U_p=p^\top r$, the raw preference-weighted reward;
- improvement over the baseline under the same preference;
- rank and normalized utility;
- worst-axis reward, variance, and Gini coefficient;
- status and distance from $p$.

The rows with `method=Cert` and `status=CERTIFICATE_ONLY` are not reward
measurements. They carry the result of the dual certificate from Phase A.

In [ ]:
# Cell 40
def build_final_table() -> pd.DataFrame:
    have_rewards = REWARD_CACHE.is_file() and REWARD_CACHE.stat().st_size > 0
    reward_by_key = {}
    if have_rewards:
        for line in REWARD_CACHE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                record = json.loads(line)
                if "key" in record and "reward" in record:  # skip header
                    reward_by_key[str(record["key"])] = np.asarray(record["reward"], dtype=np.float64)

    out = []
    for _, r in lam_df_B.iterrows():
        p = r[p_cols].to_numpy(float)
        lam = r[lam_cols].to_numpy(float)
        usable = bool(r["usable"])
        record = {
            "p_name": r["p_name"], "p": np.round(p, 4).tolist(),
            "method": r["method"], "params": r["params"], "status": r["status"],
            "lambda": np.round(lam, 4).tolist() if usable else None,
            "dist_l2": r["dist_l2"], "dist_R": r["dist_R"],
        }
        if have_rewards:
            r_p = reward_by_key.get(coefficient_key(p))
            u_base = float(preference_utility(r_p[None, :], p, "identity")[0]) if r_p is not None else np.nan
            if r["method"] == "Cert":
                u_lam, note = u_base, "certificate: lambda = p, Delta = 0 by construction"
            elif not usable:
                u_lam, note = np.nan, r["status"]
            else:
                r_l = reward_by_key.get(coefficient_key(lam))
                u_lam = float(preference_utility(r_l[None, :], p, "identity")[0]) if r_l is not None else np.nan
                note = "" if r_l is not None else "reward missing"
            record.update({"U_p(p)": u_base, "U_p(lambda)": u_lam,
                           "Delta U_p": u_lam - u_base, "note": note})
        else:
            record.update({"U_p(p)": np.nan, "U_p(lambda)": np.nan, "Delta U_p": np.nan,
                           "proxy_pRlam": r["proxy_pRlam"],
                           "note": "Phase B was not run"})
        out.append(record)
    return pd.DataFrame(out)


final_df = build_final_table()
final_df.to_csv(FINAL_CSV, index=False)
print(f"Final table -> {FINAL_CSV}  ({len(final_df)} rows)")
with pd.option_context("display.max_rows", 250, "display.width", 240):
    display(final_df.round(5))

**Cell 41**

### 11.1 Condensed View by Method

The empty fields for `CERTIFICATE_ONLY` are intentional. A proxy certificate is
not a reward measurement; mixing both would violate Wall A.

In [ ]:
# Cell 42
agg = {"n": ("dist_l2", "size"),
       "n_usable": ("lambda", lambda s: int(s.notna().sum())),
       "mean_dist_l2": ("dist_l2", "mean"),
       "mean_dist_R": ("dist_R", "mean")}
if final_df["Delta U_p"].notna().any():
    agg.update({"mean_Delta_U_p": ("Delta U_p", "mean"),
                "n_improved": ("Delta U_p", lambda s: int((s > 0).sum())),
                "n_worse": ("Delta U_p", lambda s: int((s < 0).sum()))})
grouped = final_df.assign(
    key=final_df["method"] + final_df["params"].map(lambda s: f"({s})" if s else "")
).groupby("key").agg(**agg)
display(grouped.round(5))

**Cell 43**

## 12. Normalization Robustness

ArmoRM axes have native scales that differ by up to a factor of three. The primary
metric therefore remains the raw preference-weighted sum. Rank and min-max
normalization are reported as sensitivity analyses on the **same reward matrix**;
they require no additional model evaluations.

In [ ]:
# Cell 44
if REWARD_CACHE.is_file() and REWARD_CACHE.stat().st_size > 0:
    _cache_records = [json.loads(line) for line in
                      REWARD_CACHE.read_text(encoding="utf-8").splitlines() if line.strip()]
    matrix = np.asarray([record["reward"] for record in _cache_records
                         if "reward" in record], dtype=np.float64)

    robust_rows = []
    for pname, p in PREF_SET_B:
        agreement = normalization_agreement(matrix, p)
        robust_rows.append({
            "p_name": pname,
            "argmax_agrees": agreement["argmax_agrees"],
            **{f"argmax_{k}": v for k, v in agreement["argmax_index"].items()},
            **{f"rho_{k}": v for k, v in agreement["spearman"].items()},
        })
    robust_df = pd.DataFrame(robust_rows)
    robust_df.to_csv(ROBUST_CSV, index=False)
    display(robust_df.round(4))
    n_agree = int(robust_df["argmax_agrees"].sum())
    print(f"\nargmax agrees for {n_agree}/{len(robust_df)} preferences across all three normalizations.")
    print("If it agrees throughout, the scale objection to the raw sum is answered empirically;")
    print("if it differs, this must be reported as a limitation, not used as a reason to switch metrics.")
else:
    print("No reward cache — robustness comparison skipped.")

**Cell 45**

## 13. Uncertainty Layer and Multiplicity

The primary estimator is the per-prompt difference

$$d_j=
p^\top r_j(\lambda_{\mathrm{method}})
-p^\top r_j(\lambda_{\mathrm{baseline}}).$$

The bootstrap resamples prompts, not scalar rewards. The same resample indices are
used for all methods and preferences (common random numbers).

NB10.1 contains 18 moving method comparisons: six each for Avg, MaxMin, and Fair.
The confirmatory decision is therefore based on **Holm-adjusted paired bootstrap
p-values**:

`holm_improves = True` if and only if the one-sided raw bootstrap p-value survives
Holm correction at familywise $\alpha=0.05$.

The raw 95% confidence interval remains in the table only as a descriptive
effect-size interval and is **not** the confirmatory decision rule.

In [ ]:
# Cell 46
STATS_CSV = RESULTS_DIR / f"nb10_{RUN_TAG}_stats.csv"

if RUN_REWARD_COLLECTION:
    from src.lambda_utils import holm_adjust, lambda_key
    from src.metrics import mean_rank, paired_bootstrap_ci, selection_regret

    # Use the same key function as the deduplication in Section 6c; otherwise
    # the lookup will not find the points again.
    def _key(vec):
        return lambda_key(np.asarray(vec, dtype=float), decimals=LAMBDA_DEDUP_DECIMALS)

    point_index = {_key(pt): i for i, pt in enumerate(EVAL_POINTS)}
    stats_rows, utilities_by_method = [], {}

    for _, r in lam_df_B.iterrows():
        if not (bool(r["usable"]) and bool(r["moved"])):
            continue
        p_vec = r[p_cols].to_numpy(float)
        lam_vec = r[lam_cols].to_numpy(float)
        i_lam, i_p = point_index.get(_key(lam_vec)), point_index.get(_key(p_vec))
        if i_lam is None or i_p is None:
            raise KeyError(f"Merge point is missing from the tensor: {r['p_name']}/{r['method']}")

        boot = paired_bootstrap_ci(REWARD_TENSOR[i_lam], REWARD_TENSOR[i_p], p_vec)
        label = f"{r['method']}({r['params']})" if r["params"] else str(r["method"])
        stats_rows.append({
            "p_name": r["p_name"], "method": r["method"], "params": r["params"], "label": label,
            "U_p_baseline": float(REWARD_TENSOR[i_p].mean(axis=0) @ p_vec),
            "U_p_lambda": float(REWARD_TENSOR[i_lam].mean(axis=0) @ p_vec),
            **{k: boot[k] for k in ("delta_u_p", "ci_low", "ci_high", "excludes_zero", "p_value")},
        })
        utilities_by_method.setdefault(label, {})[r["p_name"]] = stats_rows[-1]["U_p_lambda"]
        utilities_by_method.setdefault("Baseline (lambda=p)", {})[r["p_name"]] = \
            stats_rows[-1]["U_p_baseline"]

    stats_df = pd.DataFrame(stats_rows)
    if len(stats_df):
        # ONLY Holm is confirmatory. The unadjusted CI remains as a
        # descriptive column and carries no global claim.
        stats_df["p_holm"] = holm_adjust(stats_df["p_value"].to_numpy(float))
        stats_df["holm_improves"] = (stats_df["delta_u_p"] > 0) & (stats_df["p_holm"] < 0.05)
        stats_df["holm_harms"] = (stats_df["delta_u_p"] < 0) & (stats_df["p_holm"] < 0.05)
        stats_df["significant"] = stats_df["holm_improves"] | stats_df["holm_harms"]

        common = set.intersection(*(set(v) for v in utilities_by_method.values()))
        if common:
            order = sorted(common)
            ranks = mean_rank({k: [v[n] for n in order] for k, v in utilities_by_method.items()})
            print(f"Mean rank across {len(order)} preferences (1 = best):")
            for name, value in sorted(ranks.items(), key=lambda kv: kv[1]):
                print(f"  {value:5.2f}  {name}")

        # Selection regret against the best point in the ENTIRE evaluated set, not
        # only against points generated for this preference. This is the only way to keep the reference set
        # identical for all methods and make the value match the wording
        # "best evaluated point" in the thesis.
        for pname, p_vec in PREF_SET_B:
            search_utilities = (REWARD_MATRIX @ np.asarray(p_vec, dtype=float)).tolist()
            mask = stats_df["p_name"] == pname
            if not bool(mask.any()):
                continue  # no moving pilot method for this preference
            stats_df.loc[mask, "selection_regret"] = [
                selection_regret(u, search_utilities) for u in stats_df.loc[mask, "U_p_lambda"]]
            stats_df.loc[mask, "baseline_regret"] = selection_regret(
                float(stats_df.loc[mask, "U_p_baseline"].iloc[0]), search_utilities)

        stats_df.to_csv(STATS_CSV, index=False)
        print(f"\n{len(stats_df)} comparisons, "
              f"{int(stats_df['excludes_zero'].sum())} with a CI excluding zero, "
              f"{int(stats_df['holm_improves'].sum())} better after Holm, "
              f"{int(stats_df['holm_harms'].sum())} worse after Holm.")
        display(stats_df.sort_values("delta_u_p", ascending=False).round(5))
    else:
        print("No moving lambda vectors in the Phase-B subset: under floor collapse, "
              "every method with lambda = p reduces to the baseline. This is the certificate from "
              "NB09.1, not a missing result.")
else:
    print("RUN_REWARD_COLLECTION is False — statistics skipped.")


**Cell 47**

## 14. Export

In [ ]:
# Cell 48
report = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "NB10.1 shortened method comparison (29 merge points)",
    "primary_matrix": PRIMARY_MATRIX,
    "methods_from": "src.coefficient_portfolio (no method defined in this notebook)",
    "n_preferences_phase_a": len(PREF_SET),
    "n_preferences_phase_b": len(PREF_SET_B),
    "regime": REGIME,
    "armorm_precision": ARMORM_PRECISION,
    "n_lambda_rows": int(len(lam_df)),
    "n_phase_b_method_rows": int(len(lam_df_B)),
    "phase_b_rule": PHASE_B_RULE,
    "scope": "shortened 29-point pilot; not a replacement for full NB10",
    "n_unique_merge_points": int(n_unique),
    "phase_b_run": bool(RUN_REWARD_COLLECTION),
    "gate_open": bool(GATE_OPEN),
    "lambda_table_sha256": lambda_hash,
    "metric_primary": "raw U_p, identity normalization",
    "cert_note": "certificate row from NB09.1 run1; not measured",
    "status_counts": lam_df["status"].value_counts().to_dict(),
}
write_json(REPORT_JSON, report)

zip_path = RESULTS_DIR / f"nb10_{RUN_TAG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (LAMBDA_CSV, FINAL_CSV, ROBUST_CSV, STATS_CSV, REPORT_JSON, PREREG_JSON,
                 REWARD_CACHE, REWARD_PROMPT_PATH):
        if path.is_file():
            archive.write(path, path.name)
print(f"{zip_path}\nSHA256 {hashlib.sha256(zip_path.read_bytes()).hexdigest()}")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass

In [ ]:
# Cell 49
!git status --short